In [22]:
import os
import cv2
import time
import torch
import torch.optim as optim
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib.pyplot as plt
import timm
import random

# MONAI imports
import monai
from monai.data import CacheDataset, DataLoader
from monai.transforms import (
    Compose,
    MapTransform,
    Resized,
    RandAffined,
    RandAdjustContrastd,
    NormalizeIntensityd,
    # CastToTyped,
    EnsureTyped,
    # CropForegroundd, 
    MaskIntensityd
)
from monai.losses import FocalLoss
from monai.utils import set_determinism
from monai.transforms import Rand2DElasticd, RandGaussianNoised
from torch.utils.data import WeightedRandomSampler
import math
from torch.utils.data import Sampler

cv2.setNumThreads(0)

def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    set_determinism(seed=seed)

set_global_seed(42)

In [23]:
# ==========================================
# 1. NATIVE MONAI CUSTOM TRANSFORMS
# ==========================================

class LoadAndApplyCLAHEd(MapTransform):
    """
    Custom MONAI dictionary transform to bypass Windows I/O limitations 
    by reading grayscale, applying CLAHE deterministically, and converting 
    to RGB before caching. Also generates a foreground mask.
    """
    def __init__(self, keys, allow_missing_keys=False):
        super().__init__(keys, allow_missing_keys)
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            img_path = d[key]
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                raise ValueError(f"CRITICAL ERROR: Failed to read image at '{img_path}'.")
            
            image_clahe = self.clahe.apply(image)
            image_rgb = cv2.cvtColor(image_clahe, cv2.COLOR_GRAY2RGB)
            image_ch_first = np.moveaxis(image_rgb, -1, 0)
            
            processed_img = image_ch_first.astype(np.float32) / 255.0
            d[key] = processed_img
            
            d["mask"] = (processed_img[0:1, ...] > 0.01).astype(np.float32)
            
        return d

In [24]:
# ==========================================
# 2. MONAI CACHE DATASET CONFIGURATION
# ==========================================

def get_combined_cv_data(root_dir, splits, classes):
    data_dicts = []
    class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
    
    for split in splits:
        split_dir = os.path.join(root_dir, split)
        if not os.path.exists(split_dir):
            continue
            
        for cls_name in classes:
            cls_dir = os.path.join(split_dir, cls_name)
            if not os.path.exists(cls_dir):
                continue
                
            for file_name in os.listdir(cls_dir):
                if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(cls_dir, file_name)
                    # Extract the patient ID before the first underscore
                    patient_id = file_name.split('_')[0]
                    
                    data_dicts.append({
                        "image": img_path, 
                        "label": class_to_idx[cls_name],
                        "group": patient_id
                    })
                    
    return data_dicts

def get_test_data_dicts(root_dir, split, classes):
    split_dir = os.path.join(root_dir, split)
    data_dicts = []
    class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
    
    for cls_name in classes:
        cls_dir = os.path.join(split_dir, cls_name)
        if not os.path.exists(cls_dir):
            continue
            
        for file_name in os.listdir(cls_dir):
            if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(cls_dir, file_name)
                data_dicts.append({"image": img_path, "label": class_to_idx[cls_name]})
                
    return data_dicts

classes = ['Biliary_Leaks', 'Lithiasis', 'Normal', 'Stricture']
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

In [25]:
train_transforms = Compose([
    LoadAndApplyCLAHEd(keys=["image"]),
    
    # Resize both image and mask
    Resized(keys=["image", "mask"], spatial_size=(512, 512), mode=["area", "nearest"]),
    
    # Rotate both image and mask
    RandAffined(keys=["image", "mask"], prob=0.8, rotate_range=15 * (np.pi / 180), 
                scale_range=0.1, translate_range=0.05, padding_mode="zeros", 
                mode=["bilinear", "nearest"]),
                
    # Warp both image and mask
    Rand2DElasticd(keys=["image", "mask"], prob=0.2, spacing=(32, 32), magnitude_range=(1, 2),
                   padding_mode="zeros", mode=["bilinear", "nearest"]),
    
    RandAdjustContrastd(keys=["image"], prob=0.5, gamma=(0.85, 1.15)),
    
    RandGaussianNoised(keys=["image"], prob=0.1, mean=0.0, std=0.02),
    
    # Apply the mask to crush background noise back to pure 0.0
    MaskIntensityd(keys=["image"], mask_key="mask"),
    
    # Dynamically normalize without ImageNet stats, ignoring the 0.0 borders
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    EnsureTyped(keys=["image", "label"])
])

val_test_transforms = Compose([
    LoadAndApplyCLAHEd(keys=["image"]),
    Resized(keys=["image", "mask"], spatial_size=(512, 512), mode=["area", "nearest"]),
    
    # Ensure resizing artifacts on the borders are masked out
    MaskIntensityd(keys=["image"], mask_key="mask"),
    
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    EnsureTyped(keys=["image", "label"])
])

In [26]:
# ==========================================
# 3. HYPERPARAMETERS & K-FOLD SETUP
# ==========================================
base_dir = './dataset'
LEARNING_RATE = 2e-5  
WEIGHT_DECAY = 1e-2   
EPOCHS = 60
WARMUP_EPOCHS = 3     
BATCH_SIZE = 4 
ACCUMULATION_STEPS = 2 
N_SPLITS = 5
PATIENCE = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Combine train and val sets and extract their groups
cv_data = get_combined_cv_data(base_dir, ['train', 'val'], classes)
cv_labels = [item["label"] for item in cv_data]
cv_groups = [item["group"] for item in cv_data]

# Setup held-out test set
test_data = get_test_data_dicts(base_dir, 'test', classes)
test_dataset = CacheDataset(data=test_data, transform=val_test_transforms, cache_rate=1.0, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# Initialize Grouped K-Fold
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

Training on device: cuda


Loading dataset: 100%|██████████| 267/267 [00:03<00:00, 88.70it/s]


In [27]:
class GuaranteedLeakBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, leak_class_idx):
        self.dataset = dataset
        self.batch_size = batch_size
        self.leak_class_idx = leak_class_idx
        
        self.labels = [item['label'] for item in dataset.data]
        self.leak_indices = [i for i, label in enumerate(self.labels) if label == self.leak_class_idx]
        self.other_indices = [i for i, label in enumerate(self.labels) if label != self.leak_class_idx]
        
        self.num_batches = math.ceil(len(self.other_indices) / (self.batch_size - 1))
        
    def __iter__(self):
        random.shuffle(self.leak_indices)
        random.shuffle(self.other_indices)
        
        leak_cycle = self.leak_indices.copy()
        batches = []
        other_idx_ptr = 0
        
        for _ in range(self.num_batches):
            batch = []
            
            if not leak_cycle:
                leak_cycle = self.leak_indices.copy()
                random.shuffle(leak_cycle)
            if leak_cycle: 
                batch.append(leak_cycle.pop())
            
            # 2. Fill the remaining slots with other classes
            while len(batch) < self.batch_size and other_idx_ptr < len(self.other_indices):
                batch.append(self.other_indices[other_idx_ptr])
                other_idx_ptr += 1
                
            random.shuffle(batch)
            batches.append(batch)
            
        random.shuffle(batches)
        return iter(batches)
        
    def __len__(self):
        return self.num_batches

In [28]:
# ==========================================
# 4. & 5. MODEL INIT & K-FOLD TRAINING LOOP
# ==========================================
for fold, (train_idx, val_idx) in enumerate(sgkf.split(cv_data, cv_labels, groups=cv_groups)):
    print(f"\n==========================================")
    print(f"========== STARTING FOLD {fold + 1}/{N_SPLITS} ==========")
    print(f"==========================================")
    
    fold_train_data = [cv_data[i] for i in train_idx]
    fold_val_data = [cv_data[i] for i in val_idx]
    
    train_dataset = CacheDataset(data=fold_train_data, transform=train_transforms, cache_rate=1.0, num_workers=0)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    
    val_dataset = CacheDataset(data=fold_val_data, transform=val_test_transforms, cache_rate=1.0, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    
    # Initialize fresh model per fold
    model = timm.create_model('convnextv2_base', pretrained=True, num_classes=4, drop_rate=0.4, drop_path_rate=0.3)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.head.parameters():
        param.requires_grad = True
    model = model.to(device)
    
    fold_train_labels = [item["label"] for item in fold_train_data]
    num_classes = len(classes)
    class_counts_array = np.bincount(fold_train_labels, minlength=num_classes)
    total_samples = len(fold_train_labels)
    
    computed_weights = total_samples / (num_classes * class_counts_array)
    weights = torch.tensor(computed_weights, dtype=torch.float32).to(device)
    
    criterion = FocalLoss(
        weight=weights, 
        gamma=3.0, 
        reduction='mean',
        to_onehot_y=True,
        use_softmax=True 
    )
    
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda')
    
    best_val_f1 = 0.0
    fold_model_save_path = f"convnextv2_base_ercp_fold{fold + 1}_best.pth"
    epochs_no_improve = 0
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        
        if epoch == WARMUP_EPOCHS:
            print(f"\n--- Unfreezing Backbone for Full Fine-Tuning (Fold {fold + 1}) ---")
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=(EPOCHS - WARMUP_EPOCHS))
        
        # --- TRAINING PHASE ---
        model.train()
        running_loss = 0.0
        train_preds, train_targets = [], []
        optimizer.zero_grad() 
        
        for i, batch in enumerate(tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1}/{EPOCHS} [Train]")):
            inputs = batch["image"].to(device)
            labels = batch["label"].to(device, dtype=torch.long)
            
            # Expand dims to satisfy MONAI's to_onehot_y logic constraints
            labels_onehot = labels.unsqueeze(-1)
            
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(inputs)
                loss = criterion(outputs, labels_onehot) 
                loss = loss / ACCUMULATION_STEPS
            
            scaler.scale(loss).backward()
            
            if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad() 
            
            running_loss += (loss.item() * ACCUMULATION_STEPS) * inputs.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_targets.extend(labels.cpu().numpy()) 
            
        train_f1 = f1_score(train_targets, train_preds, average='macro')
        epoch_loss = running_loss / len(train_loader.dataset)
        
        # --- VALIDATION PHASE ---
        model.eval()
        running_val_loss = 0.0
        val_preds, val_targets = [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Fold {fold+1} Epoch {epoch+1}/{EPOCHS} [Val]"):
                inputs = batch["image"].to(device)
                labels = batch["label"].to(device, dtype=torch.long)
                labels_onehot = labels.unsqueeze(-1)
                
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels_onehot)
                
                running_val_loss += loss.item() * inputs.size(0)
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())
        
        val_loss = running_val_loss / len(val_loader.dataset)
        val_f1 = f1_score(val_targets, val_preds, average='macro')
        
        scheduler.step()
        time_elapsed = time.time() - start_time
        print(f"Fold {fold+1} Epoch {epoch+1} | {time_elapsed:.0f}s | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), fold_model_save_path)
            print(f"--> Saved new best model for Fold {fold + 1} with Val F1: {best_val_f1:.4f}")
            print("\n--- Current Best Fold Classification Report ---")
            print(classification_report(val_targets, val_preds, target_names=classes, zero_division=0))
        else:
            epochs_no_improve += 1
            print(f"Early stopping counter: {epochs_no_improve}/{PATIENCE}")
            
        if epochs_no_improve >= PATIENCE:
            print(f"\n--- Early stopping triggered for Fold {fold + 1} at epoch {epoch + 1} ---")
            break


========== STARTING FOLD 1/5 ==========


Fold 1 Epoch 1/60 [Val]: 100%|██████████| 68/68 [00:02<00:00, 27.62it/s]


Fold 1 Epoch 1 | 37s | Train Loss: 0.1840 | Val Loss: 0.1768 | Train F1: 0.2346 | Val F1: 0.2343
--> Saved new best model for Fold 1 with Val F1: 0.2343

--- Current Best Fold Classification Report ---
               precision    recall  f1-score   support

Biliary_Leaks       0.06      0.03      0.04        40
    Lithiasis       0.45      0.68      0.54       120
       Normal       0.29      0.39      0.33        51
    Stricture       0.17      0.02      0.03        61

     accuracy                           0.38       272
    macro avg       0.24      0.28      0.23       272
 weighted avg       0.30      0.38      0.31       272



Fold 1 Epoch 2/60 [Val]: 100%|██████████| 68/68 [00:02<00:00, 27.83it/s]


Fold 1 Epoch 2 | 37s | Train Loss: 0.1724 | Val Loss: 0.1792 | Train F1: 0.2640 | Val F1: 0.2352
--> Saved new best model for Fold 1 with Val F1: 0.2352

--- Current Best Fold Classification Report ---
               precision    recall  f1-score   support

Biliary_Leaks       0.00      0.00      0.00        40
    Lithiasis       0.46      0.51      0.48       120
       Normal       0.23      0.43      0.30        51
    Stricture       0.19      0.13      0.16        61

     accuracy                           0.33       272
    macro avg       0.22      0.27      0.24       272
 weighted avg       0.29      0.33      0.30       272



...

Fold 5 Epoch 22 | 67s | Train Loss: 0.0051 | Val Loss: 0.4509 | Train F1: 0.9773 | Val F1: 0.5069
Early stopping counter: 3/5


Fold 5 Epoch 23/60 [Val]: 100%|██████████| 65/65 [00:02<00:00, 25.44it/s]


Fold 5 Epoch 23 | 69s | Train Loss: 0.0025 | Val Loss: 0.4400 | Train F1: 0.9842 | Val F1: 0.5086
Early stopping counter: 4/5


Fold 5 Epoch 24/60 [Val]: 100%|██████████| 65/65 [00:02<00:00, 25.47it/s]


Fold 5 Epoch 24 | 69s | Train Loss: 0.0027 | Val Loss: 0.5277 | Train F1: 0.9840 | Val F1: 0.4642
Early stopping counter: 5/5

--- Early stopping triggered for Fold 5 at epoch 24 ---


In [ ]:
# ==========================================
# 6. TESTING EVALUATION (ENSEMBLE)
# ==========================================
print("\n==========================================")
print("RUNNING FINAL ENSEMBLE EVALUATION ON TEST SET")
print("==========================================")

num_test_samples = len(test_dataset)
num_classes = len(classes)

# Tensor to accumulate the probability scores from all saved models
ensemble_probs = torch.zeros((num_test_samples, num_classes), device=device)
test_targets = []
targets_collected = False

# Loop through all saved fold models
for fold in range(1, N_SPLITS + 1):
    fold_model_save_path = f"convnextv2_base_ercp_fold{fold}_best.pth"
    
    # Check if fold completed and saved a model
    if not os.path.exists(fold_model_save_path):
        print(f"Skipping Fold {fold}: No model saved.")
        continue
        
    print(f"\nLoading weights from Fold {fold}: {fold_model_save_path}")
    model.load_state_dict(torch.load(fold_model_save_path))
    model.eval()
    
    current_idx = 0
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"[Testing Fold {fold}]"):
            inputs = batch["image"].to(device)
            labels = batch["label"]
            
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(inputs)
                # Convert raw logits to probabilities
                probs = torch.softmax(outputs, dim=1)
                
            batch_size = inputs.size(0)
            
            # Add this model's probabilities to the running total
            ensemble_probs[current_idx : current_idx + batch_size] += probs
            current_idx += batch_size
            
            if not targets_collected:
                test_targets.extend(labels.numpy())
                
    targets_collected = True

# Divide by the number of splits to get the exact average probability across the ensemble
ensemble_probs = ensemble_probs / N_SPLITS

# The final prediction is the class with the highest average probability
test_preds = torch.argmax(ensemble_probs, dim=1).cpu().numpy()

print("\n==========================================")
print("--- Final Ensemble Classification Report ---")
print("==========================================")
print(classification_report(test_targets, test_preds, target_names=classes, zero_division=0))

test_f1 = f1_score(test_targets, test_preds, average='macro')
print(f"Final Ensemble Macro F1: {test_f1:.4f}")


RUNNING FINAL ENSEMBLE EVALUATION ON TEST SET

Loading weights from Fold 1: convnextv2_base_ercp_fold1_best.pth


[Testing Fold 1]: 100%|██████████| 67/67 [00:02<00:00, 26.75it/s]



Loading weights from Fold 2: convnextv2_base_ercp_fold2_best.pth


[Testing Fold 2]: 100%|██████████| 67/67 [00:02<00:00, 26.49it/s]



Loading weights from Fold 3: convnextv2_base_ercp_fold3_best.pth


...


Loading weights from Fold 4: convnextv2_base_ercp_fold4_best.pth


[Testing Fold 4]: 100%|██████████| 67/67 [00:02<00:00, 27.01it/s]



Loading weights from Fold 5: convnextv2_base_ercp_fold5_best.pth


[Testing Fold 5]: 100%|██████████| 67/67 [00:02<00:00, 29.01it/s]


--- Final Ensemble Classification Report ---
               precision    recall  f1-score   support

Biliary_Leaks       0.67      0.12      0.20        17
    Lithiasis       0.72      0.84      0.77       123
       Normal       0.53      0.72      0.61        43
    Stricture       0.87      0.64      0.74        84

     accuracy                           0.71       267
    macro avg       0.70      0.58      0.58       267
 weighted avg       0.73      0.71      0.70       267

Final Ensemble Macro F1: 0.5813
